# REINFORCE：最朴素的策略梯度算法
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：REINFORCE.py | 核心功能：蒙特卡洛策略梯度、回报标准化、PyTorch 策略网络

## 概述

REINFORCE 是最基础的策略梯度算法。与 Q-Learning/SARSA 学习价值函数不同，REINFORCE 直接优化策略函数 pi(a|s;theta)。

核心思想：如果一个动作带来了高回报，就增加它被选中的概率；反之则降低。数学上基于策略梯度定理：梯度 J(theta) = E[梯度 log pi(a|s) x G_t]。

REINFORCE 是蒙特卡洛方法——必须等整个 episode 结束才能计算回报 G_t 并更新。这导致高方差，但低偏差。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 数学原理

### 1. 策略梯度定理

**代码对应**：REINFORCE 直接优化策略 $\pi(a|s; \theta)$。

目标函数：$J(\theta) = \mathbb{E}_{\pi_\theta}\left[\sum_{t=0}^{\infty}\gamma^t r_t\right]$

**策略梯度定理**：

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[\sum_{t=0}^{\infty}\nabla_\theta\ln\pi_\theta(a_t|s_t) \cdot G_t\right]$$

其中 $G_t = \sum_{k=t}^{\infty}\gamma^{k-t}r_k$ 为从时刻 $t$ 开始的**回报**（return）。

### 2. REINFORCE 算法

**蒙特卡洛策略梯度**：用采样轨迹估计期望：

$$\theta \leftarrow \theta + \alpha\sum_{t=0}^{T}\nabla_\theta\ln\pi_\theta(a_t|s_t) \cdot G_t$$

流程：
1. 用当前策略 $\pi_\theta$ 采样完整轨迹 $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$
2. 计算每个时刻的回报 $G_t$
3. 更新 $\theta$

### 3. 基线（Baseline）减小方差

**代码对应**：引入基线 $b(s)$（如状态值函数 $V(s)$）减小方差：

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\nabla_\theta\ln\pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))\right]$$

$b(s)$ 不改变梯度的期望（无偏），但显著减小方差。常用 $b(s) = V(s; \phi)$。

### 4. 优缺点

- **优点**：可处理连续动作空间，策略梯度是无偏的
- **缺点**：方差大（蒙特卡洛采样），收敛慢
- **改进**：Actor-Critic 用 TD 误差代替蒙特卡洛回报，减小方差

### 1. 简单 CartPole 环境

运行 1. 简单 CartPole 环境 的代码，观察算法在该环节的行为。

In [ ]:
class SimpleCartPole:
    def __init__(self):
        self.max_steps = 200
        self.reset()

    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, 4)
        self.steps = 0
        return self.state.copy()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 1.0 if action == 1 else -1.0
        x_ddot = (force - 0.0025 * x_dot + 0.001 * np.sin(theta)) / 1.0
        theta_ddot = (0.01 * np.sin(theta) - 0.001 * theta_dot + 0.0001 * force) / 0.1
# --- 赋值/配置 ---
        dt = 0.02
        x_dot += x_ddot * dt
        theta_dot += theta_ddot * dt
        x += x_dot * dt
        theta += theta_dot * dt
# --- 数值计算 ---
        self.state = np.array([x, x_dot, theta, theta_dot])
        self.steps += 1
        done = abs(x) > 2.4 or abs(theta) > 0.21 or self.steps >= self.max_steps
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

### 2. 策略网络

运行 2. 策略网络 的代码，观察算法在该环节的行为。

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim=4, action_dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
# --- 继续 ---
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
            nn.Softmax(dim=-1),
# --- 继续 ---
        )

    def forward(self, x):
        return self.net(x)

### 3. REINFORCE 算法

运行 3. REINFORCE 算法 的代码，观察算法在该环节的行为。

In [ ]:
class REINFORCEAgent:
    def __init__(self, state_dim=4, action_dim=2, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.policy = PolicyNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
# --- 赋值/配置 ---
        self.log_probs = []
        self.rewards = []

    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs = self.policy(state_t)
        dist = Categorical(probs)
        action = dist.sample()
# --- 继续 ---
        self.log_probs.append(dist.log_prob(action))
        return action.item()

    def store_reward(self, reward):
        self.rewards.append(reward)

    def update(self):
        # 计算折扣回报
        returns = []
        G = 0
        for r in reversed(self.rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
# --- 继续 ---
        returns = torch.FloatTensor(returns)

        # 标准化回报（减少方差）
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 策略梯度损失: -E[log π(a|s) × G]
        loss = 0
        for log_prob, G in zip(self.log_probs, returns):
            loss -= log_prob * G

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.log_probs = []
        self.rewards = []

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = SimpleCartPole()
agent = REINFORCEAgent()

print("=== REINFORCE 训练 ===")
n_episodes = 500
reward_history = []

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        state, reward, done = env.step(action)
        agent.store_reward(reward)
        total_reward += reward

    agent.update()
    reward_history.append(total_reward)

    if (episode + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"  Episode {episode+1:>3}: 平均奖励={avg:>6.1f}")

### 5. 测试

运行 5. 测试 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 测试（无探索）===")
test_rewards = []
for _ in range(10):
    state = env.reset()
    total_reward = 0
# --- 赋值/配置 ---
    done = False
    while not done:
        state_t = torch.FloatTensor(state)
        probs = agent.policy(state_t)
        action = probs.argmax().item()
# --- 继续 ---
        state, reward, done = env.step(action)
        total_reward += reward
    test_rewards.append(total_reward)
print(f"10 次测试平均奖励: {np.mean(test_rewards):.1f}")

### 6. REINFORCE 原理

运行 6. REINFORCE 原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== REINFORCE 原理 ===")
print("策略梯度定理: ∇J(θ) = E[∇log π(a|s;θ) × G_t]")
print("  π(a|s;θ): 给定状态 s 下选择动作 a 的概率")
print("  G_t: 从 t 时刻起的折扣累积回报")
print("  更新方向：增加高回报动作的概率，降低低回报动作的概率")

print("\n=== REINFORCE 要点 ===")
print("- On-policy：需要从当前策略采样数据")
print("- 蒙特卡洛方法：等 episode 结束后才更新（高方差）")
print("- 回报标准化可减少方差：(G - mean(G)) / std(G)")
print("- 优点：可处理连续动作空间、理论保证收敛")
# --- 输出结果 ---
print("- 缺点：高方差、收敛慢（相比 Actor-Critic）")
print("- 改进：加 baseline（如价值函数）减少方差 → Actor-Critic")

## 关键代码解释

### 折扣回报计算

```python
G = 0
for r in reversed(self.rewards):
    G = r + self.gamma * G
    returns.insert(0, G)
```

从后往前累加折扣回报。这是蒙特卡洛方法的标志——用完整的 episode 回报。

### 回报标准化

```python
returns = (returns - returns.mean()) / (returns.std() + 1e-8)
```

标准化减少方差，是 REINFORCE 的标准实践。

### 策略梯度损失

```python
loss -= log_prob * G  # 增加高回报动作的概率
```

## 注意事项

1. **高方差**：蒙特卡洛采样方差大，需要很多 episode 才能收敛
2. **On-policy**：每次更新后旧数据作废
3. **回报标准化很重要**：否则训练极不稳定

## 总结与延伸

以上代码展示了 **REINFORCE** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **加 baseline 的 REINFORCE**：用 V(s) 做 baseline 减少方差，这就引出了 Actor-Critic
- **Entropy Bonus**：鼓励探索，防止策略过早收敛
- **VPG（Vanilla Policy Gradient）**：REINFORCE 的通用框架版本
